# 08 RAG Evaluation Dashboard
Calcul des métriques d'évaluation RAG sur le dernier run.

In [ ]:
summary_sql = '''
WITH latest_run AS (
    SELECT max(executed_at) AS executed_at
    FROM fr_raise.rag_pipeline.rag_evaluation
),
base AS (
    SELECT *
    FROM fr_raise.rag_pipeline.rag_evaluation
    WHERE executed_at = (SELECT executed_at FROM latest_run)
),
coverage AS (
    SELECT
        COUNT(DISTINCT CASE WHEN retrieval_success = 1 THEN expected_category END) AS covered_categories,
        COUNT(DISTINCT expected_category) AS total_categories
    FROM base
)
SELECT
    COUNT(*) AS total_questions,
    AVG(CAST(retrieval_success AS DOUBLE)) * 100.0 AS retrieval_success_rate,
    AVG(response_time_ms) AS avg_response_time,
    AVG(CAST(retrieval_count AS DOUBLE)) AS avg_chunks_retrieved,
    CASE WHEN total_categories = 0 THEN 0.0 ELSE (covered_categories / total_categories) * 100.0 END AS category_coverage,
    AVG(answer_quality) AS avg_answer_quality,
    AVG(faithfulness_score) AS avg_faithfulness_score,
    AVG(relevance_score) AS avg_relevance_score
FROM base
CROSS JOIN coverage
'''
display(spark.sql(summary_sql))

In [ ]:
display(spark.sql('SELECT question_id, expected_category, retrieval_success, retrieval_count, response_time_ms, answer_quality FROM fr_raise.rag_pipeline.rag_evaluation ORDER BY executed_at DESC, question_id'))